# ASR A/B preference rating (Colab / Workbench)

1. Set **RATER_ID** below
2. **Run All**
3. Pick **A / B / Tie / Skip** for each clip

Data: `gs://dd_tfx_full_transcripts/transcripts/` + `audio/` (+ `asr/dd210/` for model JSONs).

See [COLAB.md](COLAB.md) for lead setup.

In [ ]:
# --- edit these ---
RATER_ID = "Alice"
GCS_BUCKET = "dd_tfx_full_transcripts"
MOUNT_POINT = "/content/dd_tfx"  # Workbench: e.g. "/home/jupyter/gcs/dd_tfx_full_transcripts"
REPO_ROOT = "/content/ab_prefs_rating"
REPO_URL = "https://github.com/rosyvs/ab_prefs_rating.git"
ASR_SUBDIR = "asr/dd210"

In [ ]:
import subprocess, sys
from pathlib import Path

repo = Path(REPO_ROOT)
if not (repo / "pyproject.toml").is_file():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
# install deps before any ab_prefs_interface import (scoring.py imports jiwer at load time)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from ab_prefs_interface.colab_setup import bootstrap
from ab_prefs_interface.launch import launch_rating

repo, session_config = bootstrap(
    RATER_ID,
    bucket=GCS_BUCKET,
    mount_point=MOUNT_POINT,
    repo_root=repo,
    repo_url=REPO_URL,
    asr_subdir=ASR_SUBDIR,
)
launch_rating(RATER_ID, session_config_path=session_config)

In [ ]:
# Optional: upload ratings to GCS
# !gsutil cp {REPO_ROOT}/results/ab_prefs/preferences_{RATER_ID}.json gs://{GCS_BUCKET}/ab_prefs/